# Notebook 06 — Deployment Capstone: Docker, GitOps & Full-Stack RAG

## Objectives
- Containerize a RAG agent with Docker
- Automate deployments with GitHub Actions (GitOps)
- Walk through the `enterprise-app/` full-stack RAG application
- Generate a production docker-compose.yml
- Review the complete 5-day learning journey

## 1. Health check endpoint

In [ ]:
# The simplest possible API: a health check
def health_check() -> dict:
    return {"status": "ok", "version": "1.0.0", "service": "rag-agent"}

def status_check(docs_loaded: int, model: str) -> dict:
    return {
        "status": "ready",
        "docs_loaded": docs_loaded,
        "model": model,
        "endpoints": ["/health", "/status", "/ask", "/stream"]
    }

print(health_check())
print(status_check(42, "gpt-4o-mini"))

## Docker Containerization

A minimal `Dockerfile` for our RAG agent:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose FastAPI port
EXPOSE 8000

# Start server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

Build and run:
```bash
docker build -t rag-agent:latest .
docker run -p 8000:8000 --env-file .env rag-agent:latest
```

## 2. Generate docker-compose.yml

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from day5.deployment import generate_docker_compose, DockerConfig

services = [
    DockerConfig(
        service_name="backend",
        image="rag-agent:latest",
        port_host=8000,
        port_container=8000,
        env_vars=["OPENAI_API_KEY=${OPENAI_API_KEY}", "LANGCHAIN_TRACING_V2=true"],
        healthcheck_path="/health"
    ),
    DockerConfig(
        service_name="frontend",
        image="rag-ui:latest",
        port_host=8501,
        port_container=8501,
        env_vars=["BACKEND_URL=http://backend:8000"],
        depends_on=["backend"]
    ),
]

compose_yaml = generate_docker_compose(services)
print(compose_yaml)

## 3. GitOps with GitHub Actions

In [ ]:
from day5.deployment import show_gitops_pattern

print(show_gitops_pattern())

## 4. The Enterprise RAG App (`enterprise-app/`)

The `enterprise-app/` folder contains a complete, production-ready RAG application.
It demonstrates all Day 5 concepts in a real codebase.

In [ ]:
import sys, os
sys.path.insert(0, '../enterprise-app')

# Show what's in the enterprise app
files = [
    "app.py          — FastAPI backend: /ask, /health, /status",
    "ui.py           — Streamlit frontend",
    "rag_pipeline.py — BM25 + OpenAI embeddings + web search fallback",
    "Dockerfile      — Container for backend",
    "docker.yml      — docker-compose for backend + frontend",
    "data/           — Knowledge base (company_handbook.md, product_faq.txt)",
]
print("Enterprise App Structure:")
for f in files:
    print(f"  {f}")

print("\nTo run locally:")
print("  cd enterprise-app")
print("  cp ../.env.example .env  # add OPENAI_API_KEY")
print("  pip install -r requirements.txt")
print("  uvicorn app:app --reload --port 8000")
print("  # In another terminal:")
print("  BACKEND_URL=http://127.0.0.1:8000 streamlit run ui.py")

print("\nTo run with Docker:")
print("  cd enterprise-app")
print("  docker-compose -f docker.yml up --build")

## 5. Production docker-compose with all services

In [ ]:
from day5.deployment import generate_docker_compose, DockerConfig

# Full production stack
services = [
    DockerConfig(
        service_name="rag-backend",
        image="enterprise-rag-backend:latest",
        port_host=8000,
        port_container=8000,
        env_vars=[
            "OPENAI_API_KEY=${OPENAI_API_KEY}",
            "LANGCHAIN_API_KEY=${LANGCHAIN_API_KEY}",
            "LANGCHAIN_TRACING_V2=true",
            "LANGCHAIN_PROJECT=enterprise-rag-prod",
        ],
        healthcheck_path="/health"
    ),
    DockerConfig(
        service_name="rag-frontend",
        image="enterprise-rag-frontend:latest",
        port_host=8501,
        port_container=8501,
        env_vars=["BACKEND_URL=http://rag-backend:8000"],
        depends_on=["rag-backend"]
    ),
]

yaml = generate_docker_compose(services)
print(yaml)

# Optionally save it
with open("../enterprise-app/generated-docker-compose.yml", "w") as f:
    f.write(yaml)
print("\nSaved to enterprise-app/generated-docker-compose.yml")

## 6. 5-Day Course Retrospective

You have built a complete enterprise RAG stack from scratch:

In [ ]:
retrospective = [
    ("Day 1", "LangChain Foundations",         ["LangChain chains", "prompt templates", "LCEL", "OpenAI + Anthropic"]),
    ("Day 2", "RAG Fundamentals",              ["document loaders", "text splitters", "embeddings", "vector stores", "FAISS", "Chroma"]),
    ("Day 3", "Advanced RAG",                  ["hybrid search", "reranking", "multi-query", "contextual compression", "RAG evaluation"]),
    ("Day 4", "Agents & LangGraph",            ["tool calling", "ReAct agents", "memory", "HITL", "MCP", "LangGraph", "multi-agent"]),
    ("Day 5", "Enterprise RAG & Deployment",   ["LangGraph orchestration", "RRF", "BM25", "SSE streaming", "RAGAS", "cost tracking", "Docker", "GitOps", "LangSmith"]),
]

print("=" * 60)
print("  5-Day LLM Engineering Bootcamp — Complete")
print("=" * 60)
for day, title, topics in retrospective:
    print(f"\n{day}: {title}")
    print(f"  Topics: {', '.join(topics[:4])}{'...' if len(topics) > 4 else ''}")

print("\n" + "=" * 60)
print("What you can build now:")
print("  - Production RAG apps with hybrid BM25 + semantic search")
print("  - Multi-agent LangGraph pipelines with parallel execution")
print("  - Streaming FastAPI backends + Streamlit/React frontends")
print("  - Automated quality monitoring with RAGAS + LangSmith")
print("  - Containerized deployments with Docker + GitHub Actions")
print("=" * 60)